[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyneuro/single-cell-tone-shock/blob/main/LA_PNmodel_MultipleInputs.ipynb)

# Single‑Neuron Calibration for the LA Disinhibition Model (Step 1)

This notebook implements **Step 1** of the LA modeling roadmap:  
building and validating a **single 3‑compartment lateral amygdala principal neuron (PN)**  
before moving to network‑level simulations.

---

## Learning objectives

By the end of this notebook, you should understand:

- Why single‑neuron validation must come before network models
- How CS synapses drive PN firing without inducing learning
- Why **Ca²⁺ signals**, not spikes alone, define plasticity thresholds
- How to identify *invalid models* early and cheaply

## Step A. Install and load NEURON

We use the **Python interface to NEURON**, a standard simulator for
biophysically detailed neuron models.

NEURON allows us to:
- represent membrane compartments explicitly
- include ionic currents and synapses
- measure voltages and synaptic currents

In [1]:
RunningInCOLAB = 'google.colab' in str(get_ipython())  # checks to see if we are in google colab
if RunningInCOLAB:                                     # installs packages if in colab
    %pip install ipywidgets==7.7.1 &> /dev/null
    %pip install neuron==8.2.4 &> /dev/null
    # clone dir
    !git clone https://github.com/cyneuro/single-cell-tone-shock.git &> /dev/null
    # change dir
    %cd single-cell-tone-shock
    # compile mod files 
    !nrnivmodl modfiles/

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider
from neuron import h

In [3]:
# loads our mod files and cell templates 
h.load_file("stdrun.hoc")
h.load_file('cells.hoc')

1.0

## Step B. Build a reduced 3‑compartment PN model

We approximate a lateral amygdala principal neuron using **three compartments**:

- **Soma** – spike initiation
- **Proximal dendrite** – CS synapse integration
- **Distal dendrite** – Ca²⁺‑rich integration zone

This is the *minimal* structure that supports:
- realistic EPSP summation
- dendritic Ca²⁺ signals
- correct firing thresholds

Anything simpler risks hiding critical biophysics.

In [4]:
# Create compartments
soma = h.Section(name="soma")
prox = h.Section(name="prox")
dist = h.Section(name="dist")

# Connect them
prox.connect(soma(1))
dist.connect(prox(1))

# Set passive properties
for sec in [soma, prox, dist]:
    sec.L = 30
    sec.diam = 20 if sec is soma else 3
    sec.cm = 1
    sec.Ra = 100
    sec.insert("pas")
    sec.g_pas = 1e-4
    sec.e_pas = -65

## Step C. Add spike‑generating currents

We add classical Hodgkin–Huxley Na⁺/K Hz range**We add classical Hodgkin–Huxley Na⁺/K⁺ channels to the soma.
observed in LA principal neurons during CS and CS–US pairing.

In [5]:
soma.insert("hh")

soma

## Step D. Create CS → PN synapses (AMPA + NMDA)

We now add glutamatergic synapses representing the conditioned stimulus (CS).

At this stage:
- ❌ No inhibition
- ❌ No shock (US)
- ❌ No plasticity updates

**Critical requirement:**  
CS input alone must *never* induce learning‑level Ca²⁺ signals.


In [6]:
# 1. Define Synapses using compiled MOD mechanisms
# Carry-over targets from Exp2Syn version:
# - AMPA: tau1=0.5, tau2=2.0, e=0
# - NMDA: tau1=2.0, tau2=80.0, effective baseline ratio ~0.001/0.002 = 0.5
cs_syn = h.AMPA_NMDA_STP_LTP(prox(0.5))
cs_syn.e = 0
cs_syn.tau_r_AMPA = 0.5
cs_syn.tau_d_AMPA = 2.0
cs_syn.tau_r_NMDA = 2.0
cs_syn.tau_d_NMDA = 80.0

# AMPA/NMDA amplitude alignment:
# this mechanism has fixed internal NMDA_ratio=0.71, so we scale gmax_NMDA
# to approximate old effective NMDA:AMPA ratio of 0.5.
cs_syn.gmax_AMPA = 0.001 * 6
cs_syn.gmax_NMDA = 0.000704 * 6

# 2. Define Inhibitory Synapses (PV + SOM)
gaba_pv = h.GABA_A_STP(prox(0.5))
gaba_pv.e_GABAA = -70
gaba_pv.tau_r_GABAA = 0.1
gaba_pv.tau_d_GABAA = 10.0
gaba_pv.Use = 1.0
gaba_pv.Dep = 0.0
gaba_pv.Fac = 0.0
gaba_pv.u0 = 1.0

gaba_som = h.GABA_A_STP(dist(0.5))
gaba_som.e_GABAA = -70
gaba_som.tau_r_GABAA = 2.0
gaba_som.tau_d_GABAA = 50.0
gaba_som.Use = 1.0
gaba_som.Dep = 0.0
gaba_som.Fac = 0.0
gaba_som.u0 = 1.0

# 3. Define Stimulators (three independent spike trains)
cs_stim = h.NetStim(); cs_stim.start = 50; cs_stim.number = 100
pv_stim = h.NetStim(); pv_stim.start = 0; pv_stim.number = 100; pv_stim.interval = 100
som_stim = h.NetStim(); som_stim.start = 0; som_stim.number = 100; som_stim.interval = 100

# 4. Define NetCons (the connections the sliders control)
nc_cs = h.NetCon(cs_stim, cs_syn)
nc_pv = h.NetCon(pv_stim, gaba_pv)
nc_som = h.NetCon(som_stim, gaba_som)

# 5. Define Recording Vectors
v_vec = h.Vector().record(soma(0.5)._ref_v)
t_vec = h.Vector().record(h._ref_t)
weighted_i = h.Vector().record(cs_syn._ref_i)
i_nmda = h.Vector().record(cs_syn._ref_i_NMDA)

## **Step E. Transition to Population Dynamics**
In the lateral amygdala, a "Tone" (CS) or "Shock" (US) involves the convergence of many individual axons.

We are moving from a **Single-Synapse** model to an **Afferent Stream** model. We will now use **Multipliers** ($N$) to simulate the total number of active inputs:
* **Total Glutamatergic Conductance:** $G_{CS} = (w_{ampa} + w_{nmda}) \times N_{CS}$
* **Total Inhibitory Conductance:** $G_{Inh} = (w_{gaba}) \times N_{Inh} \times (1 - \text{VIP})$

This allows us to see how the neuron integrates a "population" of inputs rather than just one.

## **Model Specification: Multi-Input PN with Disinhibitory Gating**

### **I. Geometry & Biophysics**
The membrane dynamics follow the cable equation:
$$C_m \frac{dV}{dt} = - \sum I_{ion} + \frac{1}{R_a} \frac{\partial^2V}{\partial x^2}$$

| Compartment | Biophysics | Target Inputs |
| :--- | :--- | :--- |
| **Soma** | Active $Na^+/K^+$ | Spike Initiation |
| **Proximal** | Passive | CS (Tone) & PV (Inh) |
| **Distal** | Passive | SOM (Inh) |

### **II. Synaptic Population Logic**
The total conductance ($G$) for a specific input stream is the product of the base weight ($w$), the number of inputs ($N$), and the kinetics ($e$):
$$G_{total}(t) = N_{inputs} \times w \times (e^{-t/\tau_2} - e^{-t/\tau_1})$$

**The Disinhibition Mechanism:**
The "Shock" (VIP) acts as a scalar that suppresses the weight of the inhibitory populations:
$$W_{effective} = W_{baseline} \times (1 - VIP_{level})$$
* $VIP = 0.0$: Full inhibition (Inhibitory population at 100% strength).
* $VIP = 1.0$: Full disinhibition (Inhibitory population silenced).

## **Problem Statement: Finding the Plasticity Threshold**

In this simulation, we explore a core question of amygdala physiology: **How do sensory inputs (CS) and neuromodulation (US/Shock) combine to trigger learning?**

### **The Goal**
Your task is to identify the **LTP Boundary**. Under normal conditions, a Principal Neuron (PN) is heavily inhibited by local interneurons (PV and SOM cells), which prevents a Tone (CS) from inducing Long-Term Potentiation (LTP).

You must determine the specific "gating" threshold where **VIP-mediated disinhibition** (the "Shock") sufficiently releases the brake on the neuron to allow $Ca^{2+}$ entry to exceed our threshold:
> $$\theta_{LTP} = 0.015$$

---

### **How the Interactive Explorer Works**

This tool uses a **Reactive Parameter Sweep** to let you test different conditions in real-time. Here is how the underlying model responds to your inputs:

* **CS Frequency (The Tone):** Controlled by the `cs_hz` slider. This adjusts the firing rate of glutamatergic inputs.
* **VIP Level (The Shock):** Controlled by the `vip_level` slider (0.0 to 1.0).
    * At **0.0**, inhibition is at 100% strength (No Shock).
    * At **1.0**, inhibition is completely silenced (Maximum Shock effect).
* **Calcium Proxy:** We use the area under the NMDA current curve (integrated via the Trapezoidal Rule) to estimate total $Ca^{2+}$ entry.

### **Visual Feedback Guide**
The explorer generates two synchronized plots to help you analyze the results:

1.  **Voltage Plot (Top):** Displays the raw electrophysiology. Look for whether the cell is just "flirting" with the threshold or firing robustly.
2.  **NMDA Current (Bottom):** The **shaded blue area** represents the $Ca^{2+}$ proxy.
    * **The Red Dashed Line** represents the goal. If the blue peaks don't cross this line, the signal will likely fail to induce LTP.

---

**Next Step:** Use the sliders in the cell below to find the minimum combination of Tone and Shock required to reach **STATUS: ✅ LTP INDUCED**.

In [7]:
# --- 1. Simulation Setup Function ---
# This function runs every time you move a slider
def run_interactive_model(cs_hz, vip_level, n_cs, n_pv, n_som):
    # Update weights based on Multipliers and VIP level
    # Baseline effective scaling: CS=0.002, GABA=0.005
    nc_cs.weight[0] = 0.002 * n_cs

    # VIP Scales the inhibitory population weight (Disinhibition)
    disinhibition = 1.0 - vip_level
    nc_pv.weight[0] = 0.005 * n_pv * disinhibition
    nc_som.weight[0] = 0.005 * n_som * disinhibition

    # Update Frequency of the Tone
    cs_stim.interval = 1000.0 / cs_hz

    # Run simulation
    h.tstop = 400
    h.run()

    # Calculate peak weighted current as the calcium proxy
    # Using np.array to ensure compatibility with matplotlib
    t_np = np.array(t_vec)
    i_np = np.array(weighted_i)
    ca_proxy = np.max(np.abs(i_np))
    ltp_threshold = 0.015 * (1.0 - 0.5 * vip_level)

    # --- 2. Visualization ---
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

    # Top Plot: Voltage (Spiking behavior)
    ax1.plot(t_np, np.array(v_vec), color='black', lw=1)
    ax1.set_ylabel("Voltage (mV)")
    ax1.set_title(f"CS: {n_cs} inputs @ {cs_hz}Hz | VIP: {int(vip_level*100)}% Disinhibition")

    # Bottom Plot: Weighted NMDA Proxy
    ax2.plot(t_np, np.abs(i_np), color='blue', lw=1)
    ax2.axhline(ltp_threshold, color='red', linestyle='--', label='LTP Threshold')
    ax2.set_ylabel("Weighted NMDA Current (Proxy)")
    ax2.set_xlabel("Time (ms)")
    ax2.legend(loc='upper right')

    # Status Indicator
    status = "✅ LTP INDUCED" if ca_proxy > ltp_threshold else "❌ NO LTP"
    plt.figtext(0.5, 0.02, f"Peak Calcium Signal: {ca_proxy:.4f} | Threshold: {ltp_threshold:.4f} | {status}",
                ha="center", fontsize=12, fontweight='bold', bbox={"facecolor":"white", "alpha":0.8, "pad":5})
    plt.show()

# --- 3. Create the Sliders ---
# Default values are set to a lower-CS / higher-shock case that still crosses the threshold.
interact(run_interactive_model,
         cs_hz=IntSlider(min=1, max=50, step=1, value=20, description='CS Freq (Hz)'),
         vip_level=FloatSlider(min=0, max=1, step=0.1, value=1.0, description='VIP (Shock)'),
         n_cs=IntSlider(min=1, max=20, step=1, value=10, description='# CS Inputs'),
         n_pv=IntSlider(min=1, max=20, step=1, value=2, description='# PV Inputs'),
         n_som=IntSlider(min=1, max=20, step=1, value=2, description='# SOM Inputs'))

plt.show()

interactive(children=(IntSlider(value=20, description='CS Freq (Hz)', max=50, min=1), FloatSlider(value=1.0, d…